# Security Issues Prioritization

Pipeline: raw findings (CSV) → LLM normalization → ROSI scoring → ranked backlog + executive report.

LLM backend: local Ollama (`llama3.1:8b` by default). Set `OLLAMA_MODEL` / `OLLAMA_HOST` in `.env` to override.

In [ ]:
# Cell 1 — Imports and setup
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

SRC_DIR = Path("src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import normalizer
import scorer
import reporter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

print(f"Using LLM model: {normalizer.OLLAMA_MODEL} @ {normalizer.OLLAMA_HOST}")

In [ ]:
# Cell 2 — Load raw findings
issues_df = pd.read_csv("data/sample_issues.csv")
display(issues_df.head())
print(f"Loaded {len(issues_df)} issues for processing")

In [ ]:
# Cell 3 — Normalize each finding with the LLM
import json
from tqdm.auto import tqdm

normalized_results: list[dict] = []
for row in tqdm(issues_df.to_dict(orient="records"), desc="Normalizing"):
    norm = normalizer.normalize_finding(row["raw_description"], row["source_tool"])
    merged = {**row, **norm}
    normalized_results.append(merged)

print("\nSample normalized output (first issue):")
print(json.dumps(normalized_results[0], indent=2, default=str))

In [ ]:
# Cell 4 — Score every normalized finding
scored_results = [scorer.score_issue(n) for n in normalized_results]

results_df = pd.DataFrame(scored_results)
display_cols = [
    "finding_id", "risk_domain", "severity",
    "asset_value_usd", "ale", "residual_ale",
    "estimated_remediation_cost_usd", "financial_benefit",
    "rosi", "priority_tier",
]
display_cols = [c for c in display_cols if c in results_df.columns]
display(results_df[display_cols].sort_values("rosi", ascending=False, na_position="last"))

In [ ]:
# Cell 5 — Generate the terminal + CSV + markdown report
reporter.generate_report(scored_results)

ranked_inline = (
    results_df.dropna(subset=["rosi"])
    .sort_values("rosi", ascending=False)
    .reset_index(drop=True)
)
ranked_inline.insert(0, "rank", ranked_inline.index + 1)
display(ranked_inline[["rank", "finding_id", "risk_domain", "severity",
                       "ale", "estimated_remediation_cost_usd",
                       "rosi", "priority_tier"]])

In [ ]:
# Cell 6 — Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

viz_df = results_df.dropna(subset=["rosi", "ale", "financial_benefit"]).copy()

tier_order = [
    "TIER 1 \u2014 IMMEDIATE",
    "TIER 2 \u2014 HIGH PRIORITY",
    "TIER 3 \u2014 STANDARD",
    "TIER 4 \u2014 MONITOR",
    "TIER 5 \u2014 DEPRIORITIZE",
]
tier_palette = {
    "TIER 1 \u2014 IMMEDIATE":     "#b30000",
    "TIER 2 \u2014 HIGH PRIORITY": "#e34a33",
    "TIER 3 \u2014 STANDARD":      "#fc8d59",
    "TIER 4 \u2014 MONITOR":       "#fdbb84",
    "TIER 5 \u2014 DEPRIORITIZE":  "#bdbdbd",
}

# --- Chart 1: ALE by risk domain ---
ale_by_domain = (
    viz_df.groupby("risk_domain")["ale"].sum()
    .sort_values(ascending=False)
)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=ale_by_domain.values, y=ale_by_domain.index, ax=ax, color="#3182bd")
ax.set_title("Total ALE by Risk Domain")
ax.set_xlabel("Annualized Loss Expectancy (USD)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()

# --- Chart 2: Remediation cost vs financial benefit (bubble=ROSI, color=tier) ---
fig, ax = plt.subplots(figsize=(10, 6))
for tier in tier_order:
    sub = viz_df[viz_df["priority_tier"] == tier]
    if sub.empty:
        continue
    ax.scatter(
        sub["estimated_remediation_cost_usd"],
        sub["financial_benefit"],
        s=(sub["rosi"].clip(lower=0) + 1) * 25,
        c=tier_palette[tier],
        alpha=0.75,
        edgecolor="black",
        linewidth=0.5,
        label=tier,
    )
ax.set_xscale("log")
ax.set_title("Remediation Cost vs. Financial Benefit (bubble size = ROSI)")
ax.set_xlabel("Remediation Cost (USD, log scale)")
ax.set_ylabel("Financial Benefit (USD)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"${y:,.0f}"))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.axhline(0, color="black", linewidth=0.6, linestyle="--", alpha=0.5)
ax.legend(title="Priority Tier", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# --- Chart 3: Top 10 issues by ROSI ---
top10 = viz_df.sort_values("rosi", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 6))
colors = [tier_palette.get(t, "#bdbdbd") for t in top10["priority_tier"]]
ax.barh(top10["finding_id"], top10["rosi"], color=colors, edgecolor="black", linewidth=0.4)
ax.invert_yaxis()
ax.set_title("Top 10 Findings by ROSI")
ax.set_xlabel("ROSI (multiple)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}x"))
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=tier_palette[t]) for t in tier_order
]
ax.legend(legend_handles, tier_order, title="Priority Tier",
          bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Export confirmation
csv_path = reporter.CSV_PATH.resolve()
md_path = reporter.MD_PATH.resolve()
print("Outputs written:")
print(f"  CSV:      {csv_path}")
print(f"  Markdown: {md_path}")